# CSE 144 Final Project — Kaggle runner (thin wrapper)

1-click Kaggle entry point. All logic lives in `src/*.py` (reproducible, version-controlled);
this notebook only: installs deps, points the code at the mounted competition data, trains, and
ensembles → `submission.csv`.

**Incremental build order (each layer gated on a measured OOF improvement):**
1. **ConvNeXt-S 5-fold → baseline OOF**  ← *run this first, in isolation*
2. + TTA → 3. + MixUp/CutMix+EMA+label-smoothing → 4. + EffV2-S & ViT ensemble → 5. + seed averaging

Do **not** run the full ensemble until the ConvNeXt-S baseline OOF is logged and approved.
Seed + every hyperparameter are in `config.yaml`. See `README.md` for env/versions/CLI.

### 1. Get the code + install timm
`torch`/`torchvision` come from the Kaggle base image — pin only `timm`.

In [ ]:
# Replace <user>/<repo> with the public GitHub repo URL.
!git clone https://github.com/<user>/<repo>.git /kaggle/working/proj
%cd /kaggle/working/proj
!pip install -q timm==1.0.27

### 2. Point config at the mounted competition data
Kaggle mounts the data read-only under `/kaggle/input/<slug>/`. Symlink it to `Data/` so the configured relative paths resolve unchanged.

In [ ]:
import os, glob
COMP = glob.glob('/kaggle/input/*'); print('inputs:', COMP)
SRC = COMP[0]                                   # adjust if multiple inputs attached
if not os.path.exists('Data'): os.symlink(SRC, 'Data')
print('train dirs:', len(glob.glob('Data/train/*')), '| test files:', len(glob.glob('Data/test/*')))

### 3. STEP 1 — ConvNeXt-S 5-fold baseline (run this first, alone)
Trains all 5 folds of ConvNeXt-S only. Saves per-fold raw+EMA ckpts + OOF logits to
`checkpoints/convnext_small/`. The training log prints the leak-free **OOF accuracy** — that is the
baseline every later layer is measured against.

In [ ]:
!python src/train.py --backbone convnext_small --device cuda

### 3b. Read the baseline OOF accuracy explicitly
Log this number to `info/changelog.md` before adding any further layer.

In [ ]:
import numpy as np
z = np.load('checkpoints/convnext_small/oof.npz')
cov = z['covered']
oof_acc = float((z['logits'][cov].argmax(1) == z['labels'][cov]).mean())
print(f'ConvNeXt-S 5-fold BASELINE OOF accuracy: {oof_acc:.4f} over {int(cov.sum())} samples')

---
### 4. LATER — full ensemble (only after the baseline is approved)
Train the remaining backbones, then ensemble all folds × backbones with hflip TTA into a validated
`submission.csv`. Each addition should be gated on an OOF improvement (see build order).

In [ ]:
# import yaml
# cfg = yaml.safe_load(open('config.yaml'))
# for bk in cfg['ensemble']:
#     if bk == 'convnext_small': continue        # already trained above
#     print('==== training', bk, '===='); !python src/train.py --backbone {bk} --device cuda
#
# !python src/predict.py --out submission.csv
# import pandas as pd; sub = pd.read_csv('submission.csv'); print(sub.shape); sub.head()

### 5. Submit
`Submit to Competition` button on this notebook's output, or the Kaggle API below.

In [ ]:
# !kaggle competitions submit -c ucsc-cse-144-spring-2026-final-project -f submission.csv -m "timm 21k ensemble + 5-fold + TTA"